# 00 Data and SGM Preparation

## Purpose

This notebook prepares the data artifacts used by the PBSMT experiments. 

In [1]:
from pathlib import Path
import subprocess, re, shutil, os, difflib

BASE_DIR = Path('/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein')
G2P_DIR = BASE_DIR / 'g2p-par'
SGM_DIR = G2P_DIR / 'sgm'
MOSES_SRC_DIR = Path('/home/kyalkalay/tools/mosesdecoder')
MOSES_SCRIPT_DIR = MOSES_SRC_DIR / 'scripts'
MOSES_BIN_DIR = MOSES_SRC_DIR / 'bin'
EXPERIMENT_PERL = MOSES_SCRIPT_DIR / 'ems' / 'experiment.perl'
MTEVAL = MOSES_SCRIPT_DIR / 'generic' / 'mteval-v13a.pl'

# The original assignment experiment directory.
ORIGINAL_EXP_DIR = BASE_DIR / 'pbsmt-big-normalize'
ORIGINAL_TEMPLATE = ORIGINAL_EXP_DIR / 'config.baseline'
ORIGINAL_BASELINE_DIR = ORIGINAL_EXP_DIR / 'baseline'

# New clean experiment directory.
EXP_DIR = BASE_DIR / 'pbsmt_original_clean_experiments'

print('BASE_DIR:', BASE_DIR)
print('G2P_DIR:', G2P_DIR)
print('SGM_DIR:', SGM_DIR)
print('Original template:', ORIGINAL_TEMPLATE, ORIGINAL_TEMPLATE.exists())
print('Clean experiment dir:', EXP_DIR)

BASE_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein
G2P_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par
SGM_DIR: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/sgm
Original template: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt-big-normalize/config.baseline True
Clean experiment dir: /media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/pbsmt_original_clean_experiments


## Check cleaned parallel files

In [2]:
required = [
    G2P_DIR/'train_clean.my', G2P_DIR/'train_clean.ph',
    G2P_DIR/'dev_clean.my', G2P_DIR/'dev_clean.ph',
    G2P_DIR/'test_clean.my', G2P_DIR/'test_clean.ph',
]
for p in required:
    print(p, 'exists=', p.exists())
    if p.exists():
        print('  lines:', sum(1 for _ in p.open(encoding='utf-8')))
assert all(p.exists() for p in required), 'Missing cleaned train/dev/test files.'

/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/train_clean.my exists= True
  lines: 20000
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/train_clean.ph exists= True
  lines: 20000
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/dev_clean.my exists= True
  lines: 2000
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/dev_clean.ph exists= True
  lines: 2000
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/test_clean.my exists= True
  lines: 2802
/media/kyalkalay/Data/AI-Class/assignment-6_aung_hein/g2p-par/test_clean.ph exists= True
  lines: 2802


## Regenerate SGM using original-style text handling

In [3]:
SGM_DIR.mkdir(parents=True, exist_ok=True)

def safe_seg_text(line):
    # Moses mteval can read raw apostrophe in segment text. Keep apostrophe unchanged.
    # Escape only angle brackets and bare ampersands that are not already entities.
    line = line.replace('&amp;', '&')
    line = re.sub(r'&(?![A-Za-z]+;|#[0-9]+;|#x[0-9A-Fa-f]+;)', '&amp;', line)
    line = line.replace('<', '&lt;').replace('>', '&gt;')
    return line

def write_sgm(input_file, output_file, lang, mode):
    lines = input_file.read_text(encoding='utf-8').splitlines()
    with output_file.open('w', encoding='utf-8') as f:
        if mode == 'src':
            f.write('<srcset setid="Myanmar_G2P_v2" srclang="any">\n')
            f.write(f'<doc docid="exp_1_normalized" genre="8000" origlang="{lang}">\n')
        elif mode == 'ref':
            f.write(f'<refset trglang="{lang}" setid="Myanmar_G2P_v2" srclang="any">\n')
            f.write('<doc sysid="ref" docid="exp_1_normalized" genre="8000" origlang="any">\n')
        else:
            raise ValueError(mode)

        for i, line in enumerate(lines, start=1):
            f.write(f'<seg id="{i}">{safe_seg_text(line)} </seg>\n')

        if mode == 'src':
            f.write('</doc>\n</srcset>\n')
        else:
            f.write('</doc>\n</refset>\n')

for lang in ['my', 'ph']:
    write_sgm(G2P_DIR/f'test_clean.{lang}', SGM_DIR/f'test.{lang}.src.sgm', lang, 'src')
    write_sgm(G2P_DIR/f'test_clean.{lang}', SGM_DIR/f'test.{lang}.ref.sgm', lang, 'ref')

for p in sorted(SGM_DIR.glob('test.*.sgm')):
    text = p.read_text(encoding='utf-8', errors='ignore')
    print(p.name, 'lines=', len(text.splitlines()), 'has_apos_entity=', '&#x27;' in text or '&apos;' in text or '&amp;#x27;' in text)

test.my.ref.sgm lines= 2806 has_apos_entity= False
test.my.src.sgm lines= 2806 has_apos_entity= False
test.ph.ref.sgm lines= 2806 has_apos_entity= False
test.ph.src.sgm lines= 2806 has_apos_entity= False


## Compare generated reference text against the original evaluation reference

In [4]:
def extract_seg_texts(sgm_file):
    texts = []
    for line in sgm_file.read_text(encoding='utf-8', errors='ignore').splitlines():
        m = re.search(r'<seg id="\d+">(.*?)\s*</seg>', line)
        if m:
            texts.append(m.group(1).strip())
    return texts

orig_ref_txt = ORIGINAL_BASELINE_DIR / 'my-ph' / 'evaluation' / 'test.reference.txt.1'
new_ref_sgm = SGM_DIR / 'test.ph.ref.sgm'

if orig_ref_txt.exists():
    orig = orig_ref_txt.read_text(encoding='utf-8', errors='ignore').splitlines()
    new = extract_seg_texts(new_ref_sgm)
    same = sum(a.strip() == b.strip() for a, b in zip(orig, new))
    print('original reference vs regenerated SGM extracted text:', same, '/', min(len(orig), len(new)))
    for i, (a, b) in enumerate(zip(orig, new), start=1):
        if a.strip() != b.strip():
            print('first difference line:', i)
            print('original:', a)
            print('new:', b)
            break
else:
    print('Original evaluation reference not found; skipped comparison:', orig_ref_txt)

original reference vs regenerated SGM extracted text: 2802 / 2802
